In [1]:
import json
from sklearn.metrics import cohen_kappa_score
# Round 1
annotated_ali = "data_annotated_80_ali.json"
annotated_wilder = "data_annotated_80_wilder.json"

with open(annotated_ali, 'r', encoding='utf-8') as file:
    ali = json.load(file)


with open(annotated_wilder, 'r', encoding='utf-8') as file:
    wilder = json.load(file)

In [2]:
TYPES = ["Lexical", "Syntactic", "Semantic", "Vagueness", "Incompleteness", "Referential"]

for t in TYPES:
    a = [ali[i][t] for i in range(len(ali))]
    w = [wilder[i][t] for i in range(len(wilder))]

    agree = sum(x == y for x, y in zip(a, w)) / len(ali)
    k = cohen_kappa_score(a, w)
    print(f"{t:<15} kappa={k:6.3f}  agreement={agree:.2%} ali_pos={sum(a):2d}  wilder_pos={sum(w):2d}")

Lexical         kappa=   nan  agreement=100.00% ali_pos= 0  wilder_pos= 0
Syntactic       kappa= 0.000  agreement=98.86% ali_pos= 1  wilder_pos= 0
Semantic        kappa= 0.000  agreement=98.86% ali_pos= 0  wilder_pos= 1
Vagueness       kappa= 0.733  agreement=87.50% ali_pos=30  wilder_pos=35
Incompleteness  kappa= 0.651  agreement=96.59% ali_pos= 6  wilder_pos= 3
Referential     kappa= 0.656  agreement=97.73% ali_pos= 4  wilder_pos= 2


/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


In [3]:
# align by id instead of trusting file order
ali_by_id = {r["id"]: r for r in ali}
wilder_by_id = {r["id"]: r for r in wilder}
ids = sorted(set(ali_by_id) & set(wilder_by_id))
disagreements = {t: [i for i in ids if ali_by_id[i][t] != wilder_by_id[i][t]] for t in TYPES}
disagreements

{'Lexical': [],
 'Syntactic': [129],
 'Semantic': [69],
 'Vagueness': [61, 62, 75, 94, 95, 102, 131, 132, 133, 134, 145],
 'Incompleteness': [65, 89, 90],
 'Referential': [71, 104]}

In [4]:
for i in ids:
    diffs = [t for t in TYPES if bool(ali_by_id[i][t]) != bool(wilder_by_id[i][t])]
    if not diffs:
        continue
    print(f"id: {i}")
    print(f"requirement: {ali_by_id[i]['description']}")
    for t in diffs:
        who = "Ali" if ali_by_id[i][t] else "Wilder"
        print(f"  - {t}: True by {who}")
    for name, rec in (("Ali", ali_by_id[i]), ("Wilder", wilder_by_id[i])):
        note = (rec.get("notes") or rec.get("note") or "").strip()
        if note:
            print(f"  note ({name}): {note}")
    print()

id: 61
requirement: The system shall allow incidental uses of protected health information (PHI) only when the underlying use or disclosure is itself permitted or required.
  - Vagueness: True by Ali
  note (Ali): incidental is vague

id: 62
requirement: The system shall allow incidental disclosures of protected health information (PHI) only when the underlying use or disclosure is itself permitted or required.
  - Vagueness: True by Ali
  note (Ali): incidental is vague

id: 65
requirement: The system must allow individuals to request restrictions on the use or disclosure of their protected health information for health care operations.
  - Incompleteness: True by Ali
  note (Ali): restriction is not clear what it is

id: 69
requirement: The system shall maintain a connection to receive PHI amendment notifications from external covered entities and apply any updates to the local designated record set upon transmission.
  - Semantic: True by Wilder

id: 71
requirement: The system must 